In [ ]:
import sensor, image, time
from pyb import UART
import ustruct

# 初始化摄像头
sensor.reset()
sensor.set_pixformat(sensor.GRAYSCALE)  
sensor.set_framesize(sensor.QQVGA)       
sensor.skip_frames(time=5000)           # 等待摄像头稳定
sensor.set_auto_gain(False)             # 关闭自动增益
sensor.set_auto_whitebal(False)         # 关闭白平衡

uart = UART(3, 115200)
BLACK_THRESHOLD = (0, 30)  # 黑色范围

def sliding_window_filter(flag0, flag1, flag2, flag3, flag4, flag5, flag6, flag7, center_x, center_y, window_size, filter_param):
    window = []  # 滑动窗口
    # 滑动窗口大小 window_size 指的是用于滤波的窗口的长度，即滑动窗口中保留的数据个数
    # filter_param 较大的滤波参数意味着更强烈的滤波效果，会使滤波结果对噪声等不良数据更不敏感，较大的滤波参数将使滤波器更加强烈地抑制高频噪声，并产生更平坦、更平滑的输出信号。

    for data in [flag0, flag1, flag2, flag3, flag4, flag5, flag6, flag7, center_x, center_y]:
        window.append(data)  # 将当前数据添加到滑动窗口中

        if len(window) > window_size:
            window.pop(0)  # 如果滑动窗口大小超过设定的窗口大小，则移除最早进入窗口的数据

    # 对窗口内的数据进行滤波处理，这里仍假设采用简单的加权平均滤波
    filtered_value = sum(window) / len(window) * filter_param
    return filtered_value


def process_rectangle(rect):
    """处理检测到的矩形"""
    # 获取矩形的四个角坐标
    corners = rect.corners()
    print("Detected Rectangle Corners:", corners)
    
    # 滑动窗口滤波处理
    center_x = center_y = 0  # 默认值
    flag0 = flag1 = flag2 = flag3 = flag4 = flag5 = flag6 = flag7 = 0
    for i, p in enumerate(corners):
        P0, P1 = p[0], p[1]
        if i == 0:
            flag0, flag1 = P0, P1
        elif i == 1:
            flag2, flag3 = P0, P1
        elif i == 2:
            flag4, flag5 = P0, P1
        elif i == 3:
            flag6, flag7 = P0, P1

    # 滑动窗口滤波的处理
    filtered_center_x = sliding_window_filter(flag0, flag1, flag2, flag3, flag4, flag5, flag6, flag7, center_x, center_y, window_size=5, filter_param=0.8)

    # 绘制矩形的四个角
    for pt in corners:
        img.draw_circle(pt[0], pt[1], 5, color=(255, 0, 0))  # 红色圆圈
    
    # 绘制矩形的边界
    for i in range(4):
        next_pt = corners[(i + 1) % 4]  # 获取下一个顶点，闭合矩形
        img.draw_line(corners[i][0], corners[i][1], next_pt[0], next_pt[1], color=127)
    
    # 通过串口发送坐标
    uart.write("Points:")
    for pt in corners:
        uart.write("%d,%d;" % (pt[0], pt[1]))
    uart.write("\n")
    
    # 打印平滑后的中心坐标
    print("Filtered Center Coordinates:", filtered_center_x)


while True:
    img = sensor.snapshot()
    
    # 应用高斯模糊进行图像平滑
    img = img.gaussian(1)  # 这里设置高斯模糊的大小为 5x5，启用反锐化操作
    
    # 使用find_rects方法查找图像中的矩形
    bin_img = img.binary([BLACK_THRESHOLD], invert=True)  # 反转二值化，使黑色区域变为白色
    rects = bin_img.find_rects(threshold=10000)  # threshold根据需要调整
    
    # 如果检测到矩形
    if rects:
        for rect in rects:
            process_rectangle(rect)  # 处理每一个检测到的矩形

In [ ]:
import sensor, image, time, math
from pyb import UART

sensor.reset()
sensor.set_pixformat(sensor.GRAYSCALE)  # 灰度图处理
sensor.set_framesize(sensor.QVGA)       # 320x240分辨率
sensor.skip_frames(time=2000)           # 等待摄像头稳定
sensor.set_auto_gain(False)             # 关闭自动增益
sensor.set_auto_whitebal(False)         # 关闭白平衡

uart = UART(3, 9600)  # 初始化UART3，波特率9600
#定义二值化范围
BLACK_THRESHOLD = (0, 70)  # 黑色阈值
def order_points(pts):#点排序
    # 按照点的顺序排序
     rect=[None]*4
     s = pts.sum(axis=1)
     rect[0] =  pts[pts.argmin()]  # X+Y 最小
     rect[2] =  pts[pts.argmax()]  # X+Y 最大
     d = [(p[0]-rect[0][0])**2 + (p[1]-rect[0][1])**2 for p in pts] # 距离左上点距离
     rect[1] = pts[d.index(max(d))]  # 右上
     rect[3] = pts[3 - d.index(max(d))]  # 左下 (剩余点)
     return rect
while( True):
     img.snapshot()  # 获取图像
     bin_img = img.binary([BLACK_THRESHOLD])  # 二值化处理
     contours = bin_img.find_contours()  # 查找轮廓
     largest_rect = None
     max_area = 0
    
     for contour in contours:
        # 计算轮廓面积
        area = contour.area()
        if area < 1000:  # 忽略小面积噪点
            continue
            
        # 多边形逼近
        approx = contour.approx(epsilon=0.02 * contour.arc_length())
        
        # 寻找四边形
        if len(approx) == 4:
            rect_pts = [(p[0], p[1]) for p in approx]
            
            # 检查凸四边形
            if contour.is_convex():
                # 更新最大面积矩形
                if area > max_area:
                    max_area = area
                    largest_rect = rect_pts
    
    # 处理找到的矩形
     if largest_rect:
        # 点排序
        ordered_pts = order_points(largest_rect)
        
        # 绘制轮廓和顶点
        for i in range(4):
            img.draw_circle(ordered_pts[i][0], ordered_pts[i][1], 5, color=127)
            next_pt = ordered_pts[(i+1)%4]
            img.draw_line(ordered_pts[i][0], ordered_pts[i][1], 
                          next_pt[0], next_pt[1], color=127)
        
        # 通过串口发送坐标
        # 格式："x0,y0,x1,y1,x2,y2,x3,y3\n"
        uart.write("Points:")
        for p in ordered_pts:
            uart.write("%d,%d;" % (p[0], p[1]))
        uart.write("\n")
     
   
    
   
   




In [ ]:
import sensor, image, time, math
from pyb import UART

# 初始化摄像头
sensor.reset()
sensor.set_pixformat(sensor.GRAYSCALE)  
sensor.set_framesize(sensor.QVGA)       
sensor.skip_frames(time=2000)           # 等待摄像头稳定
sensor.set_auto_gain(False)             # 关闭自动增益
sensor.set_auto_whitebal(False)         # 关闭白平衡

uart = UART(3, 115200)
BLACK_THRESHOLD = (0, 20)  # 0-70为黑色范围，确保这与背景区分明显

def order_points(pts):
    """按顺序排列四个点：左上，右上，右下，左下"""
    rect = [None] * 4
    
    # 去除重复点，防止重复坐标影响排序
    pts = list(set(pts))  # 去除重复的点

    if len(pts) < 4:  # 如果点数不足四个，可能是检测出错
        return pts
    
    # 计算每个点的 x + y 和
    s = [p[0] + p[1] for p in pts]  # 计算 x + y 和
    rect[0] = pts[s.index(min(s))]  # 左上 (x + y最小)
    rect[2] = pts[s.index(max(s))]  # 右下 (x + y最大)

    # 计算水平和垂直距离，并确定右上和左下
    right_up_dist = float('inf')
    left_down_dist = float('inf')

    for p in pts:
        if p != rect[0] and p != rect[2]:  # 排除左上和右下
            # 计算左上点到当前点的水平和垂直距离
            horizontal = abs(rect[0][0] - p[0])
            vertical = abs(rect[0][1] - p[1])
            
            # 如果是右上点，水平距离小，垂直距离大
            if horizontal > vertical:
                if horizontal < right_up_dist:
                    right_up_dist = horizontal
                    rect[1] = p  # 右上点
            else:
                if vertical < left_down_dist:
                    left_down_dist = vertical
                    rect[3] = p  # 左下点

    return rect

while(True):
    img = sensor.snapshot()
    bin_img = img.binary([BLACK_THRESHOLD],invert=True)  #invert=True反转
    

    #bin_img.erode(2)   # 腐蚀：消除小的噪点，避免周围区域被误识别
    blobs = bin_img.find_blobs([BLACK_THRESHOLD], merge=True)
    largest_rect = None
    max_area = 0
    
    for blob in blobs:
        # 计算每个blob的面积
        area = blob.area()
        if area < 1000:  
            continue
        rect_pts = [(blob.x(), blob.y()), 
                    (blob.x() + blob.w(), blob.y()), 
                    (blob.x() + blob.w(), blob.y() + blob.h()), 
                    (blob.x(), blob.y() + blob.h())]
    
        if area > max_area:
            max_area = area
            largest_rect = rect_pts
    
   
    if largest_rect:
        
        print("Detected Rectangle Points:", largest_rect)
        ordered_pts = order_points(largest_rect)
        print("Ordered Rectangle Points:", ordered_pts)
        
        # 绘制轮廓和顶点
        for i in range(4):
            img.draw_circle(ordered_pts[i][0], ordered_pts[i][1], 5, color=127)
            next_pt = ordered_pts[(i+1)%4]
            img.draw_line(ordered_pts[i][0], ordered_pts[i][1], 
                          next_pt[0], next_pt[1], color=127)
        
        # 通过串口发送坐标
        # 格式："x0,y0,x1,y1,x2,y2,x3,y3\n"
        uart.write("Points:")
        for p in ordered_pts:
            uart.write("%d,%d;" % (p[0], p[1]))
        uart.write("\n")

高斯版本

In [ ]:
import sensor, image, time
from pyb import UART

# 初始化摄像头
sensor.reset()
sensor.set_pixformat(sensor.GRAYSCALE)  
sensor.set_framesize(sensor.QQVGA)       
sensor.skip_frames(time=2000)           # 等待摄像头稳定
sensor.set_auto_gain(False)             # 关闭自动增益
sensor.set_auto_whitebal(False)         # 关闭白平衡

uart = UART(3, 9600)
BLACK_THRESHOLD = (0, 20)  # 黑色范围

def process_rectangle(rect):
    """处理检测到的矩形"""
    # 获取矩形的四个角坐标
    corners = rect.corners()
    print("Detected Rectangle Corners:", corners)
    
    # 绘制矩形的四个角
    for pt in corners:
        img.draw_circle(pt[0], pt[1], 5, color=(255, 0, 0))  # 红色圆圈
    
    # 绘制矩形的边界
    for i in range(4):
        next_pt = corners[(i + 1) % 4]  # 获取下一个顶点，闭合矩形
        img.draw_line(corners[i][0], corners[i][1], next_pt[0], next_pt[1], color=127)
    
    # 通过串口发送坐标
    uart.write("Points:")
    for pt in corners:
        uart.write("%d,%d;" % (pt[0], pt[1]))
    uart.write("\n")

while True:
    img = sensor.snapshot()
    
    # 应用高斯模糊进行图像平滑
    img = img.gaussian(1)  # 这里设置高斯模糊的大小为 5x5，启用反锐化操作
    
    # 使用find_rects方法查找图像中的矩形
    bin_img = img.binary([BLACK_THRESHOLD], invert=True)  # 反转二值化，使黑色区域变为白色
    rects = bin_img.find_rects(threshold=10000)  # threshold根据需要调整
    
    # 如果检测到矩形
    if rects:
        for rect in rects:
            process_rectangle(rect)  # 处理每一个检测到的矩形


加红点点版本

In [ ]:
import sensor, image, time

from pyb import Servo

from pyb import millis
from math import pi, isnan
 
class PID:
    _kp = _ki = _kd = _integrator = _imax = 0
    _last_error = _last_derivative = _last_t = 0
    _RC = 1/(2 * pi * 20)
    def __init__(self, p=0, i=0, d=0, imax=0):
        self._kp = float(p)
        self._ki = float(i)
        self._kd = float(d)
        self._imax = abs(imax)
        self._last_derivative = float('nan')
 
    def get_pid(self, error, scaler):
        tnow = millis()
        dt = tnow - self._last_t
        output = 0
        if self._last_t == 0 or dt > 1000:
            dt = 0
            self.reset_I()
        self._last_t = tnow
        delta_time = float(dt) / float(1000)
        output += error * self._kp
        if abs(self._kd) > 0 and dt > 0:
            if isnan(self._last_derivative):
                derivative = 0
                self._last_derivative = 0
            else:
                derivative = (error - self._last_error) / delta_time
            derivative = self._last_derivative + \
                                     ((delta_time / (self._RC + delta_time)) * \
                                        (derivative - self._last_derivative))
            self._last_error = error
            self._last_derivative = derivative
            output += self._kd * derivative
        output *= scaler
        if abs(self._ki) > 0 and dt > 0:
            self._integrator += (error * self._ki) * scaler * delta_time
            if self._integrator < -self._imax: self._integrator = -self._imax
            elif self._integrator > self._imax: self._integrator = self._imax
            output += self._integrator
        return output
    def reset_I(self):
        self._integrator = 0
        self._last_derivative = float('nan')

pan_servo=Servo(1)
tilt_servo=Servo(2)

pan_servo.calibration(500,2500,500)
tilt_servo.calibration(500,2500,500)

red_threshold  = (13, 49, 18, 61, 6, 47)

pan_pid = PID(p=0.07, i=0, imax=90) #脱机运行或者禁用图像传输，使用这个PID
tilt_pid = PID(p=0.05, i=0, imax=90) #脱机运行或者禁用图像传输，使用这个PID
#pan_pid = PID(p=0.1, i=0, imax=90)#在线调试使用这个PID
#tilt_pid = PID(p=0.1, i=0, imax=90)#在线调试使用这个PID

sensor.reset() # Initialize the camera sensor.
sensor.set_pixformat(sensor.RGB565) # use RGB565.
sensor.set_framesize(sensor.QQVGA) # use QQVGA for speed.
sensor.skip_frames(10) # Let new settings take affect.
sensor.set_auto_whitebal(False) # turn this off.
clock = time.clock() # Tracks FPS.

def find_max(blobs):
    max_size=0
    for blob in blobs:
        if blob[2]*blob[3] > max_size:
            max_blob=blob
            max_size = blob[2]*blob[3]
    return max_blob


while(True):
    clock.tick() # Track elapsed milliseconds between snapshots().
    img = sensor.snapshot() # Take a picture and return the image.

    blobs = img.find_blobs([red_threshold])
    if blobs:
        max_blob = find_max(blobs)
        pan_error = max_blob.cx()-img.width()/2
        tilt_error = max_blob.cy()-img.height()/2

        print("pan_error: ", pan_error)

        img.draw_rectangle(max_blob.rect()) # rect
        img.draw_cross(max_blob.cx(), max_blob.cy()) # cx, cy

        pan_output=pan_pid.get_pid(pan_error,1)/2
        tilt_output=tilt_pid.get_pid(tilt_error,1)
        print("pan_output",pan_output)
        pan_servo.angle(pan_servo.angle()+pan_output)
        tilt_servo.angle(tilt_servo.angle()-tilt_output)